In [1]:
%pip install plotly kaleido --quiet

import json
import math
import re
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import requests
import plotly.io as pio

BASE_DIR = Path.cwd()
if not (BASE_DIR / "interviews.csv").exists():
    for candidate in [Path("/Users/martateodoratrales/Desktop/AirPanel"), Path.cwd().parent]:
        if (candidate / "interviews.csv").exists():
            BASE_DIR = candidate
            break

def pick_output_dir(preferred):
    for candidate in [preferred, Path.cwd() / "models_outputs", Path("/tmp/models_outputs")]:
        try:
            candidate.mkdir(parents=True, exist_ok=True)
            probe = candidate / ".write_test"
            probe.write_text("ok")
            probe.unlink()
            return candidate
        except Exception:
            continue
    raise PermissionError("No writable output directory found")

OUTPUT_ROOT = pick_output_dir(BASE_DIR / "models_outputs")

SESSION = requests.Session()
OLLAMA_URL = "http://localhost:11434/api/chat"
OLLAMA_OPTIONS = {
    "temperature": 0.1,
    "num_predict": 220,
    "num_ctx": 4096,
    "num_gpu": -1,
}
KEEP_ALIVE = "45m"
PILOT_N = 30
CHECKPOINT_EVERY = 50
CONFIDENCE_THRESHOLD = 0.7

pio.templates.default = "plotly_white"

MODEL_SPECS = [
    {"name": "mistral:7b", "slug": "mistral_7b"},
    {"name": "gemma3", "slug": "gemma3"},
    {"name": "gemma4", "slug": "gemma4"},
]

BRAND_COLORS = {"Bouygues": "#0055A4", "Orange": "#FF6600"}
PREF_COLORS = {"Bouygues": "#0055A4", "Orange": "#FF6600", "Mixed": "#8A8A8A", "Neutral": "#D9D9D9"}
BRANDS = ["Bouygues", "Orange"]

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 160)

print(BASE_DIR)
print(OUTPUT_ROOT)



Note: you may need to restart the kernel to use updated packages.
/Users/martateodoratrales/Desktop/AirPanel
/Users/martateodoratrales/Desktop/AirPanel/models_outputs


In [2]:
def slugify(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return re.sub(r"[^\w]+", "_", s).strip("_").lower()

def extract_json(text):
    try:
        m = re.search(r"\{.*\}", text, re.DOTALL)
        return json.loads(m.group()) if m else None
    except Exception:
        return None

def normalize_panel(df):
    out = df.copy()
    if "panelist_id" not in out.columns:
        if "participant_id" in out.columns:
            out["panelist_id"] = out["participant_id"]
        elif "id" in out.columns:
            out["panelist_id"] = out["id"]
    out["panelist_id"] = out["panelist_id"].astype(str)
    if "location.citysize" in out.columns and "location.city_size" not in out.columns:
        out["location.city_size"] = out["location.citysize"]
    if "location.city_size" in out.columns and "location.citysize" not in out.columns:
        out["location.citysize"] = out["location.city_size"]
    if "incomelevel" in out.columns and "income_level" not in out.columns:
        out["income_level"] = out["incomelevel"]
    if "income_level" in out.columns and "incomelevel" not in out.columns:
        out["incomelevel"] = out["income_level"]
    if "age" in out.columns:
        out["age"] = pd.to_numeric(out["age"], errors="coerce")
        bins = [0, 24, 34, 49, 64, 120]
        labels = ["18-24", "25-34", "35-49", "50-64", "65+"]
        out["age_group"] = pd.cut(out["age"], bins=bins, labels=labels, right=True, include_lowest=True)
        out["age_group"] = out["age_group"].astype(str).replace("nan", np.nan)
    comp_cols = sorted([c for c in out.columns if c.startswith("Q_comparaison_")], key=lambda x: int(re.findall(r"(\d+)$", x)[0]))
    out["verbatim"] = out[comp_cols].fillna("").astype(str).agg(" ".join, axis=1).str.replace(r"\s+", " ", regex=True).str.strip() if comp_cols else ""
    out = out[out["verbatim"].astype(str).str.len() > 0].copy().reset_index(drop=True)
    return out

def load_source_frame(source_candidates):
    for name in source_candidates:
        path = BASE_DIR / name
        if path.exists():
            df = pd.read_csv(path)
            return normalize_panel(df), path.name
    raise FileNotFoundError(f"No source file found among: {source_candidates}")

def build_score_prompt(schema_scores):
    schema_lines = "\n".join(f"- {dim}: {rule}" for dim, rule in schema_scores.items())
    fields_lines = "\n".join(f'  "{dim}": {{"Bouygues": <0-10>, "Orange": <0-10>}},' for dim in schema_scores)
    return f'''Tu es un annotateur expert en analyse de discours publicitaire.
Tu analyses des verbatims de panélistes ayant regardé deux publicités télévisées :
- Bouygues Telecom
- Orange

Pour chaque verbatim, attribue un score de 0 à 10 à chaque marque sur chaque dimension.
0 = pas du tout / absent, 10 = extrêmement fort / dominant.

DIMENSIONS :
{schema_lines}

RÈGLES STRICTES :
1. Réponds UNIQUEMENT avec un objet JSON valide, sans texte avant ou après.
2. Chaque score est un entier entre 0 et 10.
3. Ajoute un champ "reasoning" très court.
4. Ajoute un champ "confidence" entre 0.0 et 1.0.

FORMAT DE SORTIE :
{{
{fields_lines}
  "reasoning": "explication courte",
  "confidence": 0.95
}}'''

def build_categorical_prompt(schema):
    schema_lines = "\n".join(f"- {dim}: {info['rule']}" for dim, info in schema.items())
    fields_lines = "\n".join(f'  "{dim}": "{info["values"][0]}",' for dim, info in schema.items())
    label_space = sorted({label for info in schema.values() for label in info["values"]})
    label_str = ", ".join(label_space)
    return f'''Tu es un annotateur expert en analyse de discours publicitaire.
Tu analyses des verbatims de panélistes ayant regardé deux publicités télévisées :
- Bouygues Telecom
- Orange

Tu dois annoter les dimensions ci-dessous à partir du verbatim fourni.

SCHÉMA D'ANNOTATION :
{schema_lines}

RÈGLES STRICTES :
1. Réponds UNIQUEMENT avec un objet JSON valide, sans texte avant ou après.
2. Chaque valeur doit être exactement l'une de : {label_str}.
3. Ajoute un champ "reasoning" très court.
4. Ajoute un champ "confidence" entre 0.0 et 1.0.

FORMAT DE SORTIE :
{{
{fields_lines}
  "reasoning": "explication courte",
  "confidence": 0.95
}}'''

def build_user_prompt(verbatim, few_shots):
    parts = ["EXEMPLES ANNOTÉS :"]
    for ex in few_shots:
        parts.append(f"Verbatim: {ex['verbatim']}")
        parts.append(f"Annotation: {json.dumps(ex['label'], ensure_ascii=False)}")
        parts.append("---")
    parts.append("À ANNOTER :")
    parts.append(verbatim)
    return "\n".join(parts)

def validate_score_payload(parsed, schema_scores):
    for dim in schema_scores:
        if dim not in parsed:
            parsed[dim] = {"Bouygues": -1, "Orange": -1}
        for brand in BRANDS:
            val = parsed.get(dim, {}).get(brand, -1)
            try:
                parsed[dim][brand] = max(0, min(10, int(val)))
            except Exception:
                parsed[dim][brand] = -1
    return parsed

def validate_categorical_payload(parsed, schema):
    for dim, info in schema.items():
        if parsed.get(dim) not in info["values"]:
            parsed[dim] = "ParseError"
    return parsed

DEMOGRAPHICS = {
    "gender": "Gender",
    "age_group": "Age",
    "csp": "CSP",
    "income_level": "Income level",
    "education": "Education",
    "location.city_size": "City size",
}



In [3]:
APPROACH_SPECS = [
    {
        "slug": "scores",
        "kind": "score",
        "source_candidates": ["interviews.csv"],
        "schema": {
            "creativity": "Rate how creative the ad for each brand feels (inventiveness, boldness of concept).",
            "humour": "Rate how funny or entertaining each brand's ad is.",
            "originality": "Rate how original, distinctive, and memorable each brand's ad feels.",
            "reliability_trust": "Rate how much each brand's ad inspires reliability, trust, and reassurance.",
            "intent_to_purchase": "Rate how much each brand's ad makes the panelist want to subscribe or switch.",
            "overall": "Rate the overall preference for each brand's ad, taking everything into account.",
        },
        "dim_labels": {
            "creativity": "Creativity",
            "humour": "Humour",
            "originality": "Originality",
            "reliability_trust": "Reliability / Trust",
            "intent_to_purchase": "Intent to purchase",
            "overall": "Overall",
        },
        "few_shots": [
            {
                "verbatim": "J'ai préféré celle de Bouygues. Elle était plus drôle, plus originale. Mais pour choisir un opérateur, Orange me rassure davantage sur la fiabilité.",
                "label": {
                    "creativity": {"Bouygues": 8, "Orange": 4},
                    "humour": {"Bouygues": 9, "Orange": 3},
                    "originality": {"Bouygues": 8, "Orange": 3},
                    "reliability_trust": {"Bouygues": 4, "Orange": 9},
                    "intent_to_purchase": {"Bouygues": 5, "Orange": 8},
                    "overall": {"Bouygues": 6, "Orange": 7},
                },
            },
            {
                "verbatim": "Bouygues, sans hésiter. Le concept est plus fort, plus drôle, plus original. Et au final la marque me donne aussi confiance.",
                "label": {
                    "creativity": {"Bouygues": 9, "Orange": 3},
                    "humour": {"Bouygues": 9, "Orange": 2},
                    "originality": {"Bouygues": 9, "Orange": 2},
                    "reliability_trust": {"Bouygues": 7, "Orange": 5},
                    "intent_to_purchase": {"Bouygues": 8, "Orange": 4},
                    "overall": {"Bouygues": 9, "Orange": 3},
                },
            },
            {
                "verbatim": "Orange est moins fun mais plus crédible, plus claire, et c'est elle qui me donnerait envie de choisir l'offre.",
                "label": {
                    "creativity": {"Bouygues": 4, "Orange": 5},
                    "humour": {"Bouygues": 3, "Orange": 2},
                    "originality": {"Bouygues": 4, "Orange": 4},
                    "reliability_trust": {"Bouygues": 4, "Orange": 9},
                    "intent_to_purchase": {"Bouygues": 4, "Orange": 9},
                    "overall": {"Bouygues": 4, "Orange": 8},
                },
            },
        ],
    },
    {
        "slug": "facetting_extended",
        "kind": "categorical",
        "source_candidates": ["interviews.csv"],
        "schema": {
            "creativity": {"values": ["Bouygues", "Orange", "Neutral", "Mixed"], "rule": "Which brand does the panelist prefer from a pure creativity angle: inventiveness of the concept, boldness of the idea, creative spark? Mixed = both equally creative. Neutral = no clear opinion."},
            "humour": {"values": ["Bouygues", "Orange", "Neutral", "Mixed"], "rule": "Which brand does the panelist find funnier or more entertaining? Mixed = both equally funny. Neutral = neither stands out for humour."},
            "originality": {"values": ["Bouygues", "Orange", "Neutral", "Mixed"], "rule": "Which brand is seen as more original, distinctive, surprising, or memorable in its creative execution? Mixed = both equally original. Neutral = no clear originality signal."},
            "reliability_trust": {"values": ["Bouygues", "Orange", "Neutral", "Mixed"], "rule": "Which brand inspires more reliability, seriousness, reassurance, or trust from the ad? Mixed = both equally trustworthy. Neutral = no clear signal."},
            "intent_to_purchase": {"values": ["Bouygues", "Orange", "Neutral", "Mixed"], "rule": "Which brand makes the panelist more likely to subscribe, switch, or purchase? Mixed = equal purchase intent. Neutral = neither drives intent."},
            "overallpreference": {"values": ["Bouygues", "Orange", "Neutral", "Mixed"], "rule": "Which brand is preferred overall, taking everything into account? Mixed = genuinely undecided overall. Neutral = no overall preference expressed."},
        },
        "dim_labels": {
            "creativity": "Creativity",
            "humour": "Humour",
            "originality": "Originality",
            "reliability_trust": "Reliability / Trust",
            "intent_to_purchase": "Intent to purchase",
            "overallpreference": "Overall preference",
        },
        "dim_groups": {
            "creative": ["creativity", "humour", "originality"],
            "commercial": ["reliability_trust", "intent_to_purchase"],
            "overall": ["overallpreference"],
        },
        "few_shots": [
            {
                "verbatim": "J'ai préféré celle de Bouygues. Elle était plus drôle, plus originale. Je l'ai trouvée plus inventive. Mais pour choisir un opérateur, Orange me rassure davantage sur la fiabilité et me donnerait plus envie de souscrire.",
                "label": {
                    "creativity": "Bouygues",
                    "humour": "Bouygues",
                    "originality": "Bouygues",
                    "reliability_trust": "Orange",
                    "intent_to_purchase": "Orange",
                    "overallpreference": "Mixed",
                },
            },
            {
                "verbatim": "Bouygues, sans hésiter. Le concept est plus fort, plus drôle, plus original. Et au final la marque me donne aussi confiance, donc c'est celle que je choisirais.",
                "label": {
                    "creativity": "Bouygues",
                    "humour": "Bouygues",
                    "originality": "Bouygues",
                    "reliability_trust": "Bouygues",
                    "intent_to_purchase": "Bouygues",
                    "overallpreference": "Bouygues",
                },
            },
            {
                "verbatim": "Je préfère Orange. C'est moins drôle, mais plus concret, plus crédible, plus rassurant. C'est clairement celle qui me donnerait envie de changer d'opérateur.",
                "label": {
                    "creativity": "Orange",
                    "humour": "Neutral",
                    "originality": "Neutral",
                    "reliability_trust": "Orange",
                    "intent_to_purchase": "Orange",
                    "overallpreference": "Orange",
                },
            },
        ],
    },
    {
        "slug": "multidim_preferences",
        "kind": "categorical",
        "source_candidates": ["interviews.csv", "humans_interviews.csv"],
        "schema": {
            "creative_preference": {"values": ["Bouygues", "Orange", "Neutral", "Mixed"], "rule": "Which brand does the panelist prefer from a CREATIVE or ENTERTAINMENT angle: humour, originality, storytelling, visual style? Mixed = liked both equally for creative reasons. Neutral = no clear creative opinion expressed."},
            "commercial_preference": {"values": ["Bouygues", "Orange", "Neutral", "Mixed"], "rule": "Which brand does the panelist prefer from a COMMERCIAL or TRUST angle: reliability, purchase intent, brand image, fiabilité? Mixed = balanced commercial appreciation. Neutral = no clear commercial opinion expressed."},
            "overall_preference": {"values": ["Bouygues", "Orange", "Neutral", "Mixed"], "rule": "What is the panelist's OVERALL preferred brand, taking everything into account? If they mention preferring one overall despite nuances, use that brand. Mixed = genuinely undecided overall."},
        },
        "dim_labels": {
            "creative_preference": "Creative preference",
            "commercial_preference": "Commercial preference",
            "overall_preference": "Overall preference",
        },
        "few_shots": [
            {
                "verbatim": "J'ai préféré Bouygues pour l'humour et l'originalité, mais Orange pour la fiabilité.",
                "label": {
                    "creative_preference": "Bouygues",
                    "commercial_preference": "Orange",
                    "overall_preference": "Mixed",
                },
            },
            {
                "verbatim": "Bouygues, sans hésiter. Pour son concept, pour la fiabilité aussi, et la pub est bien meilleure.",
                "label": {
                    "creative_preference": "Bouygues",
                    "commercial_preference": "Bouygues",
                    "overall_preference": "Bouygues",
                },
            },
            {
                "verbatim": "Orange est moins amusante mais plus crédible, plus rassurante et c'est elle qui me convainc vraiment.",
                "label": {
                    "creative_preference": "Orange",
                    "commercial_preference": "Orange",
                    "overall_preference": "Orange",
                },
            },
        ],
    },
    {
        "slug": "informativeness_expressivity",
        "kind": "score",
        "source_candidates": ["interviews.csv"],
        "schema": {
            "informativeness": "Rate how informative, concrete, and factually useful each brand's ad feels.",
            "expressivity": "Rate how expressive, emotional, vivid, and affectively engaging each brand's ad feels.",
        },
        "dim_labels": {
            "informativeness": "Informativeness",
            "expressivity": "Expressivity",
        },
        "few_shots": [
            {
                "verbatim": "La pub Bouygues est plus claire sur le problème du wifi dans la maison et montre concrètement la solution. Celle d'Orange est plus sobre mais me touche moins.",
                "label": {
                    "informativeness": {"Bouygues": 8, "Orange": 6},
                    "expressivity": {"Bouygues": 7, "Orange": 4},
                },
            },
            {
                "verbatim": "Orange me paraît plus informative parce que le message sur la fiabilité est plus direct. Bouygues est plus théâtrale, plus drôle, plus vivante.",
                "label": {
                    "informativeness": {"Bouygues": 6, "Orange": 9},
                    "expressivity": {"Bouygues": 9, "Orange": 5},
                },
            },
            {
                "verbatim": "Bouygues est très expressive et mémorable, mais Orange explique mieux ce qu'elle promet vraiment.",
                "label": {
                    "informativeness": {"Bouygues": 5, "Orange": 8},
                    "expressivity": {"Bouygues": 9, "Orange": 4},
                },
            },
        ],
    },
]

APPROACH_INDEX = {spec["slug"]: spec for spec in APPROACH_SPECS}
[(spec["slug"], spec["kind"]) for spec in APPROACH_SPECS]



[('scores', 'score'),
 ('facetting_extended', 'categorical'),
 ('multidim_preferences', 'categorical'),
 ('informativeness_expressivity', 'score')]

In [4]:
def call_ollama(model_name, system_prompt, user_prompt, max_retries=2):
    last_payload = None
    for attempt in range(max_retries):
        try:
            resp = SESSION.post(
                OLLAMA_URL,
                json={
                    "model": model_name,
                    "messages": [
                        {"role": "system", "content": system_prompt},
                        {"role": "user", "content": user_prompt},
                    ],
                    "stream": False,
                    "keep_alive": KEEP_ALIVE,
                    "options": OLLAMA_OPTIONS,
                },
                timeout=240,
            )
            data = resp.json()
            last_payload = data
            if "message" in data and "content" in data["message"]:
                return data["message"]["content"]
            time.sleep(1.5 * (attempt + 1))
        except Exception as e:
            last_payload = {"error": str(e)}
            time.sleep(1.5 * (attempt + 1))
    return json.dumps(last_payload, ensure_ascii=False)

def annotate_verbatim(verbatim, model_name, spec):
    if spec["kind"] == "score":
        system_prompt = build_score_prompt(spec["schema"])
    else:
        system_prompt = build_categorical_prompt(spec["schema"])
    user_prompt = build_user_prompt(verbatim, spec["few_shots"])
    generated = call_ollama(model_name, system_prompt, user_prompt)
    parsed = extract_json(generated)
    if parsed is None:
        if spec["kind"] == "score":
            out = {dim: {"Bouygues": -1, "Orange": -1} for dim in spec["schema"]}
        else:
            out = {dim: "ParseError" for dim in spec["schema"]}
        out["reasoning"] = str(generated)[:240]
        out["confidence"] = 0.0
        out["raw_output"] = str(generated)
        return out
    if spec["kind"] == "score":
        parsed = validate_score_payload(parsed, spec["schema"])
    else:
        parsed = validate_categorical_payload(parsed, spec["schema"])
    confidence = float(parsed.get("confidence", 0.0))
    for dim in spec["schema"]:
        parsed[f"{dim}_flagged"] = confidence < CONFIDENCE_THRESHOLD
    parsed["raw_output"] = generated
    return parsed

def warm_model(model_name):
    try:
        _ = call_ollama(model_name, "Réponds uniquement par OK.", "OK", max_retries=1)
        print(f"warmed {model_name}")
    except Exception as e:
        print(f"warmup failed for {model_name}: {e}")

def parse_score_columns(final_df, spec):
    out = final_df.copy()
    for dim in spec["schema"]:
        for brand in BRANDS:
            col = f"{dim}_{brand.lower()}"
            if isinstance(out[dim].iloc[0], str):
                out[col] = out[dim].apply(
                    lambda x: json.loads(str(x).replace("'", '"')).get(brand, np.nan) if pd.notna(x) else np.nan
                )
            else:
                out[col] = out[dim].apply(lambda x: x.get(brand, np.nan) if isinstance(x, dict) else np.nan)
    return out

def get_available_demographics(df):
    available = {}
    for col, label in DEMOGRAPHICS.items():
        if col in df.columns:
            available[col] = label
    return available

def prep_demo_col(df, demo_col):
    temp = df.copy()
    if demo_col == "age_group" and "age_group" not in temp.columns and "age" in temp.columns:
        temp["age"] = pd.to_numeric(temp["age"], errors="coerce")
        bins = [0, 24, 34, 49, 64, 120]
        labels = ["18-24", "25-34", "35-49", "50-64", "65+"]
        temp["age_group"] = pd.cut(temp["age"], bins=bins, labels=labels, right=True, include_lowest=True)
    temp = temp[temp[demo_col].notna()].copy()
    temp["_demo"] = temp[demo_col].astype(str)
    temp = temp[temp["_demo"].str.strip() != ""].copy()
    return temp

def save_figure(fig, path, width=1600, height=900, scale=2):
    try:
        fig.write_image(str(path), width=width, height=height, scale=scale)
    except Exception:
        fig.write_html(str(path.with_suffix(".html")))



In [5]:
def ensure_dirs(model_slug, approach_slug):
    run_dir = OUTPUT_ROOT / model_slug / approach_slug
    csv_dir = run_dir / "csv"
    plots_dir = run_dir / "plots"
    by_demo_dir = plots_dir / "by_demographic"
    hist_dir = plots_dir / "histograms"
    donut_dir = plots_dir / "donuts"
    grouped_dir = plots_dir / "grouped"
    for d in [run_dir, csv_dir, plots_dir, by_demo_dir, hist_dir, donut_dir, grouped_dir]:
        d.mkdir(parents=True, exist_ok=True)
    return {
        "run": run_dir,
        "csv": csv_dir,
        "plots": plots_dir,
        "by_demo": by_demo_dir,
        "hist": hist_dir,
        "donut": donut_dir,
        "grouped": grouped_dir,
    }

def run_annotations(df, model_name, spec, dirs):
    final_path = dirs["csv"] / "annotations_full.csv"
    checkpoint_path = dirs["csv"] / "annotations_checkpoint.csv"
    pilot_path = dirs["csv"] / "pilot_annotations.csv"

    if not pilot_path.exists():
        pilot_df = df.sample(n=min(PILOT_N, len(df)), random_state=42).copy().reset_index(drop=True)
        pilot_results = []
        for i, row in pilot_df.iterrows():
            t0 = time.time()
            result = annotate_verbatim(row["verbatim"], model_name, spec)
            result["panelist_id"] = row["panelist_id"]
            pilot_results.append(result)
            elapsed = time.time() - t0
            print(f"{model_name} | {spec['slug']} | pilot {i+1}/{len(pilot_df)} | {elapsed:.1f}s | conf={result.get('confidence', '?')}")
        pd.DataFrame(pilot_results).to_csv(pilot_path, index=False)

    if final_path.exists():
        return pd.read_csv(final_path), pilot_path

    if checkpoint_path.exists():
        checkpoint_df = pd.read_csv(checkpoint_path)
        done_ids = set(checkpoint_df["panelist_id"].astype(str))
        all_results = checkpoint_df.to_dict("records")
    else:
        done_ids = set()
        all_results = []

    remaining = df[~df["panelist_id"].astype(str).isin(done_ids)].copy().reset_index(drop=True)
    run_start = time.time()
    total = len(df)
    completed = len(all_results)

    for i, row in remaining.iterrows():
        t0 = time.time()
        result = annotate_verbatim(row["verbatim"], model_name, spec)
        result["panelist_id"] = row["panelist_id"]
        all_results.append(result)
        completed += 1
        elapsed = time.time() - t0
        if completed % CHECKPOINT_EVERY == 0:
            pd.DataFrame(all_results).to_csv(checkpoint_path, index=False)
            avg = (time.time() - run_start) / max(1, i + 1)
            eta = avg * max(0, len(remaining) - i - 1) / 60
            print(f"{model_name} | {spec['slug']} | {completed}/{total} | {elapsed:.1f}s | ETA {eta:.0f} min")

    final_df = pd.DataFrame(all_results)
    final_df.to_csv(final_path, index=False)
    return final_df, pilot_path

def save_score_outputs(panel, final_df, spec, dirs):
    final = parse_score_columns(final_df, spec)
    score_cols = [f"{dim}_{brand.lower()}" for dim in spec["schema"] for brand in BRANDS]
    cols_to_merge = ["panelist_id"] + score_cols + ["confidence", "reasoning"]
    panel_merged = panel.merge(final[cols_to_merge], on="panelist_id", how="left")
    panel_merged.to_csv(dirs["csv"] / "panel_merged.csv", index=False)

    summary = pd.DataFrame({
        "Dimension": [spec["dim_labels"].get(dim, dim) for dim in spec["schema"]],
        "Bouygues": [round(final[f"{dim}_bouygues"].mean(), 2) for dim in spec["schema"]],
        "Orange": [round(final[f"{dim}_orange"].mean(), 2) for dim in spec["schema"]],
    })
    summary.to_csv(dirs["csv"] / "summary.csv", index=False)

    fig = go.Figure()
    for brand in BRANDS:
        fig.add_trace(go.Bar(
            name=brand,
            x=[spec["dim_labels"].get(dim, dim) for dim in spec["schema"]],
            y=[round(final[f"{dim}_{brand.lower()}"].mean(), 2) for dim in spec["schema"]],
            marker_color=BRAND_COLORS[brand],
        ))
    fig.update_layout(
        barmode="group",
        title=f"{spec['slug']} | mean scores",
        yaxis=dict(range=[0, 10], title="Mean score"),
        xaxis=dict(title=""),
        width=1600,
        height=900,
    )
    save_figure(fig, dirs["plots"] / "mean_scores.png", width=1600, height=900, scale=2)

    available_demographics = get_available_demographics(panel_merged)
    for dim in spec["schema"]:
        for demo_col, demo_label in available_demographics.items():
            plot_df = prep_demo_col(panel_merged, demo_col)
            score_cols_dim = [f"{dim}_{b.lower()}" for b in BRANDS]
            long = plot_df[["_demo"] + score_cols_dim].melt(
                id_vars="_demo",
                value_vars=score_cols_dim,
                var_name="brand_col",
                value_name="score",
            )
            long["brand"] = long["brand_col"].str.replace(f"{dim}_", "", regex=False).str.capitalize()
            grouped = long.groupby(["_demo", "brand"])["score"].agg(mean_score="mean", sem=lambda x: x.sem()).reset_index()
            grouped.rename(columns={"_demo": demo_label}, inplace=True)
            n_levels = grouped[demo_label].nunique()
            n_cols = min(5, max(1, math.ceil(math.sqrt(n_levels))))
            n_rows = max(1, math.ceil(n_levels / n_cols))
            fig = px.bar(
                grouped,
                x="brand",
                y="mean_score",
                color="brand",
                facet_col=demo_label,
                facet_col_wrap=n_cols,
                error_y="sem",
                color_discrete_map=BRAND_COLORS,
                title=f"{spec['dim_labels'].get(dim, dim)} by {demo_label}",
                range_y=[0, 10],
                facet_row_spacing=0.02,
                facet_col_spacing=0.03,
            )
            fig.update_layout(showlegend=False, width=2400, height=max(1000, 420 * n_rows))
            out = dirs["by_demo"] / f"{slugify(dim)}__by__{slugify(demo_col)}.png"
            save_figure(fig, out, width=2400, height=max(1000, 420 * n_rows), scale=2)
    return panel_merged, summary

def make_grouped_categorical_plot(df, dims, label_map, group_name, demo_col, demo_label):
    temp = prep_demo_col(df, demo_col)
    rows = []
    for dim in dims:
        ct = (
            temp.groupby("_demo")[dim]
            .value_counts(normalize=True)
            .mul(100)
            .rename("pct")
            .reset_index()
        )
        ct = ct.rename(columns={"_demo": demo_label, dim: "preference"})
        ct["dimension"] = label_map.get(dim, dim)
        rows.append(ct)
    plot_df = pd.concat(rows, ignore_index=True)
    plot_df = plot_df[plot_df["preference"].isin(["Bouygues", "Orange", "Mixed", "Neutral"])].copy()
    n_levels = plot_df[demo_label].nunique()
    n_cols = min(5, max(1, math.ceil(math.sqrt(n_levels))))
    n_rows = max(1, math.ceil(n_levels / n_cols))
    fig = px.bar(
        plot_df,
        x="dimension",
        y="pct",
        color="preference",
        facet_col=demo_label,
        facet_col_wrap=n_cols,
        category_orders={"preference": ["Bouygues", "Orange", "Mixed", "Neutral"]},
        color_discrete_map=PREF_COLORS,
        title=f"{group_name} by {demo_label}",
        facet_row_spacing=0.02,
        facet_col_spacing=0.03,
    )
    fig.update_layout(width=2200, height=max(900, 420 * n_rows))
    return fig

def save_categorical_outputs(panel, final_df, spec, dirs):
    final = final_df.copy()
    cols_to_merge = ["panelist_id"] + list(spec["schema"].keys()) + ["confidence", "reasoning"]
    panel_merged = panel.merge(final[cols_to_merge], on="panelist_id", how="left")
    panel_merged.to_csv(dirs["csv"] / "panel_merged.csv", index=False)

    dims = list(spec["schema"].keys())
    summary = pd.DataFrame({
        "Dimension": [spec["dim_labels"].get(dim, dim) for dim in dims],
        "Bouygues": [100 * (final[dim] == "Bouygues").mean() for dim in dims],
        "Orange": [100 * (final[dim] == "Orange").mean() for dim in dims],
        "Mixed": [100 * (final[dim] == "Mixed").mean() for dim in dims],
        "Neutral": [100 * (final[dim] == "Neutral").mean() for dim in dims],
    }).round(2)
    summary.to_csv(dirs["csv"] / "summary.csv", index=False)

    labels_order = ["Bouygues", "Orange", "Mixed", "Neutral"]
    fig = go.Figure()
    for label in labels_order:
        vals = [(final[dim] == label).mean() * 100 for dim in dims]
        fig.add_trace(go.Bar(
            name=label,
            x=[spec["dim_labels"].get(dim, dim) for dim in dims],
            y=vals,
            marker_color=PREF_COLORS[label],
        ))
    fig.update_layout(barmode="stack", title=f"{spec['slug']} | preference distribution", width=1600, height=900)
    save_figure(fig, dirs["plots"] / "stacked_distribution.png", width=1600, height=900, scale=2)

    available_demographics = get_available_demographics(panel_merged)
    long_rows = []
    for dim in dims:
        counts = (
            final[dim]
            .value_counts(dropna=False)
            .reindex(labels_order, fill_value=0)
            .reset_index()
        )
        counts.columns = ["preference", "n"]
        fig_hist = px.bar(
            counts,
            x="preference",
            y="n",
            color="preference",
            category_orders={"preference": labels_order},
            color_discrete_map=PREF_COLORS,
            title=f"{spec['dim_labels'].get(dim, dim)} distribution",
        )
        fig_hist.update_layout(showlegend=False, width=1400, height=900)
        save_figure(fig_hist, dirs["hist"] / f"{slugify(dim)}__distribution.png", width=1400, height=900, scale=2)

        fig_donut = go.Figure(go.Pie(
            labels=labels_order,
            values=[int((final[dim] == label).sum()) for label in labels_order],
            hole=0.5,
            marker_colors=[PREF_COLORS[label] for label in labels_order],
            sort=False,
        ))
        fig_donut.update_layout(title=f"{spec['dim_labels'].get(dim, dim)} share", width=1400, height=900)
        save_figure(fig_donut, dirs["donut"] / f"{slugify(dim)}__share.png", width=1400, height=900, scale=2)

        for demo_col, demo_label in available_demographics.items():
            temp = prep_demo_col(panel_merged, demo_col)
            ct = (
                temp.groupby("_demo")[dim]
                .value_counts(normalize=True)
                .mul(100)
                .rename("pct")
                .reset_index()
            )
            ct["dimension"] = dim
            ct["dimension_label"] = spec["dim_labels"].get(dim, dim)
            ct["demographic"] = demo_col
            ct["demographic_label"] = demo_label
            ct = ct.rename(columns={"_demo": "segment", dim: "preference"})
            long_rows.append(ct)

            plot_df = ct[ct["preference"].isin(labels_order)].copy()
            plot_df = plot_df.rename(columns={"segment": demo_label})
            n_levels = plot_df[demo_label].nunique()
            n_cols = min(5, max(1, math.ceil(math.sqrt(n_levels))))
            n_rows = max(1, math.ceil(n_levels / n_cols))
            fig = px.bar(
                plot_df,
                x="preference",
                y="pct",
                color="preference",
                facet_col=demo_label,
                facet_col_wrap=n_cols,
                category_orders={"preference": labels_order},
                color_discrete_map=PREF_COLORS,
                title=f"{spec['dim_labels'].get(dim, dim)} by {demo_label}",
                facet_row_spacing=0.02,
                facet_col_spacing=0.03,
            )
            fig.update_layout(showlegend=False, width=2200, height=max(900, 420 * n_rows))
            out = dirs["by_demo"] / f"{slugify(dim)}__by__{slugify(demo_col)}.png"
            save_figure(fig, out, width=2200, height=max(900, 420 * n_rows), scale=2)

    if long_rows:
        long_table = pd.concat(long_rows, ignore_index=True)
        long_table.to_csv(dirs["csv"] / "facet_table.csv", index=False)

    if "dim_groups" in spec:
        for group_name, group_dims in spec["dim_groups"].items():
            if len(group_dims) <= 1:
                continue
            for demo_col, demo_label in available_demographics.items():
                fig = make_grouped_categorical_plot(panel_merged, group_dims, spec["dim_labels"], group_name, demo_col, demo_label)
                out = dirs["grouped"] / f"{slugify(group_name)}__facets__{slugify(demo_col)}.png"
                save_figure(fig, out, width=2200, height=1200, scale=2)

    return panel_merged, summary



In [6]:
def run_one_model_approach(model_spec, approach_spec):
    panel, source_name = load_source_frame(approach_spec["source_candidates"])
    dirs = ensure_dirs(model_spec["slug"], approach_spec["slug"])
    final_df, pilot_path = run_annotations(panel, model_spec["name"], approach_spec, dirs)

    quality_rows = []
    for dim in approach_spec["schema"]:
        flagged_col = f"{dim}_flagged"
        quality_rows.append({
            "dimension": dim,
            "parse_errors": int((final_df[dim].astype(str) == "ParseError").sum()) if approach_spec["kind"] == "categorical" else np.nan,
            "low_confidence": int(final_df[flagged_col].sum()) if flagged_col in final_df.columns else np.nan,
        })
    quality = pd.DataFrame(quality_rows)
    quality["mean_confidence"] = final_df["confidence"].mean() if "confidence" in final_df.columns else np.nan
    quality.to_csv(dirs["csv"] / "quality_report.csv", index=False)

    if approach_spec["kind"] == "score":
        panel_merged, summary = save_score_outputs(panel, final_df, approach_spec, dirs)
    else:
        panel_merged, summary = save_categorical_outputs(panel, final_df, approach_spec, dirs)

    manifest_row = {
        "model": model_spec["name"],
        "model_slug": model_spec["slug"],
        "approach": approach_spec["slug"],
        "kind": approach_spec["kind"],
        "source_csv": source_name,
        "n_rows": len(panel),
        "mean_confidence": round(float(final_df["confidence"].mean()), 4) if "confidence" in final_df.columns else np.nan,
        "output_dir": str(dirs["run"]),
        "pilot_csv": str(pilot_path),
        "final_csv": str(dirs["csv"] / "annotations_full.csv"),
        "merged_csv": str(dirs["csv"] / "panel_merged.csv"),
        "summary_csv": str(dirs["csv"] / "summary.csv"),
    }
    print(manifest_row)
    return manifest_row

def run_models(target_models=None, target_approaches=None):
    selected_models = [m for m in MODEL_SPECS if target_models is None or m["name"] in target_models or m["slug"] in target_models]
    selected_approaches = [a for a in APPROACH_SPECS if target_approaches is None or a["slug"] in target_approaches]
    manifest_rows = []
    for model_spec in selected_models:
        warm_model(model_spec["name"])
        for approach_spec in selected_approaches:
            manifest_rows.append(run_one_model_approach(model_spec, approach_spec))
    manifest = pd.DataFrame(manifest_rows)
    manifest.to_csv(OUTPUT_ROOT / "run_manifest.csv", index=False)
    return manifest



In [7]:
TARGET_MODELS = [m["name"] for m in MODEL_SPECS]
TARGET_APPROACHES = [a["slug"] for a in APPROACH_SPECS]
TARGET_MODELS, TARGET_APPROACHES



(['mistral:7b', 'gemma3', 'gemma4'],
 ['scores',
  'facetting_extended',
  'multidim_preferences',
  'informativeness_expressivity'])

In [8]:
manifest = run_models(TARGET_MODELS, TARGET_APPROACHES)
manifest



warmed mistral:7b


KeyboardInterrupt: 